In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!pip install transformers datasets evaluate accelerate scikit-learn

In [ ]:
import torch
from datasets import load_dataset
from transformers import (ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer)
from torchvision import transforms
import evaluate
import numpy as np
from PIL import Image
# from datasets import concatenate_datasets

#-------reading data
dataset = load_dataset("imagefolder", data_dir="/content/drive/MyDrive/AML/archive")
print(dataset)
print("Classes:", dataset["train"].features["label"].names)
#----------
#--------shuffling pics from folders
dataset["train"] = dataset["train"].shuffle(seed=42)
dataset["validation"] = dataset["validation"].shuffle(seed=42)
#-----------------

#--- dictionary
labels = dataset["train"].features["label"].names #getting labels
label2id = {label: i for i, label in enumerate(labels)} #translating label to number
id2label = {i: label for i, label in enumerate(labels)} #translating number to label
#---------------

#========================filtering ===========================

# 1) TRAINING PICTURES
train_labels = dataset["train"]["label"]

# we use all the pictures from boredom data set to train
boredom_idx = [i for i, l in enumerate(train_labels) if l == label2id["boredom"]]

#only 250 from the 7 remaining folders:
other_idx = [i for i, l in enumerate(train_labels) if l != label2id["boredom"]][:250*7]
dataset["train"] = dataset["train"].select(boredom_idx + other_idx)

# 2) VALIDATION PICTURES
val_labels = dataset["validation"]["label"]
#only 50 from the 7 remaining folders:

boredom_val_idx = [i for i, l in enumerate(val_labels) if l == label2id["boredom"]]

other_val_idx = [i for i, l in enumerate(val_labels) if l != label2id["boredom"]][:50*7]
dataset["validation"] = dataset["validation"].select(boredom_val_idx + other_val_idx)

print("Train:", len(dataset["train"]), "Val:", len(dataset["validation"]))

MODEL_NAME = "trpakov/vit-face-expression"
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

#flipping, rotating, lighting pics to create chaos
transforming = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), #prep size for model
])

# adding color to images
def transform_train(batch):
    images = [transforming(img.convert("RGB")) for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

def transform_val(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs


dataset["train"].set_transform(transform_train)
dataset["validation"].set_transform(transform_val)

#=================MODEL===========================

#loading ready model from hugging face
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# freezing encoder because of the model smalness (the boredom detection)
model.vit.requires_grad_(False) #makes the model to train only new head
model = model.to("cuda") #adding to cuda to use gpu ;))
accuracy_metric = evaluate.load("accuracy")

def accuracy_compute(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/AML/vit-emotion-boredom",
    num_train_epochs=10,
    train_batch_size=8, #small batch because data set for boredom is only 60 pics
    eval_batch_size=8,
    learning_rate=1e-3,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="/content/drive/MyDrive/AML/logs",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    accuracy_compute=accuracy_compute,
)

trainer.train()
#saving to:
trainer.save_model("/content/drive/MyDrive/AML/vit-emotion-final")
processor.save_pretrained("/content/drive/MyDrive/AML/vit-emotion-final")
